In [1]:
!pip -q install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 19.2 MB/s eta 0:00:00


In [2]:
!kaggle datasets download muneeburrehman98/danbooru-annotated-images

Dataset URL: https://www.kaggle.com/datasets/muneeburrehman98/danbooru-annotated-images
License(s): MIT
100%|███████████████████████████████████████| 1.64G/1.64G [00:12<00:00, 138MB/s]



In [3]:
!unzip -q danbooru-annotated-images.zip

In [4]:
import json
from pathlib import Path

def fix_coco_category_ids(json_path):
    with open(json_path, 'r') as f:
        data = json.load(f)

    for ann in data['annotations']:
        ann['category_id'] += 1

    for cat in data['categories']:
        cat['id'] += 1

    with open(json_path, 'w') as f:
        json.dump(data, f)

fix_coco_category_ids("dataset/annotations.json")
print("Category IDs adjusted successfully!")

Category IDs adjusted successfully!


In [5]:
import json
from pathlib import Path

def normalize_coco_coordinates(json_path):
    with open(json_path, 'r') as f:
        data = json.load(f)

    img_lookup = {img['id']: (img['width'], img['height']) for img in data['images']}

    for ann in data['annotations']:
        img_id = ann['image_id']
        if img_id not in img_lookup:
            continue
            
        img_w, img_h = img_lookup[img_id]
        
        x, y, w, h = ann['bbox']

        new_x = max(0, min(x, img_w))
        new_y = max(0, min(y, img_h))
        new_w = max(0, min(w, img_w - new_x))
        new_h = max(0, min(h, img_h - new_y))

        ann['bbox'] = [new_x, new_y, new_w, new_h]
        
    with open(json_path, 'w') as f:
        json.dump(data, f, indent=4)

normalize_coco_coordinates("dataset/annotations.json")
print("Coordinates clipped to image boundaries successfully!")

Coordinates clipped to image boundaries successfully!


In [6]:
import json
from pathlib import Path
from sklearn.model_selection import train_test_split

dataset_dir = "dataset"
seed = 42

dataset_path = Path(dataset_dir)
master_json = dataset_path / "annotations.json"

train_json = dataset_path / "train_annotations.json"
val_json = dataset_path / "val_annotations.json"
test_json = dataset_path / "test_annotations.json"

if train_json.exists() and val_json.exists() and test_json.exists():
    print("Splits already exist")
    exit()

with open(master_json, 'r') as f:
    data = json.load(f)

images = data['images']
annotations = data['annotations']
categories = data['categories']

buckets = [img.get('bucket', 'unknown') for img in images]

train_imgs, temp_imgs, train_buckets, temp_buckets = train_test_split(
    images, buckets, test_size=0.30, stratify=buckets, random_state=seed
)

val_imgs, test_imgs = train_test_split(
    temp_imgs, test_size=0.50, stratify=temp_buckets, random_state=seed
)

train_ids = {img['id'] for img in train_imgs}
val_ids = {img['id'] for img in val_imgs}
test_ids = {img['id'] for img in test_imgs}

train_data = {
    "images": train_imgs,
    "annotations": [ann for ann in annotations if ann['image_id'] in train_ids],
    "categories": categories
}

val_data = {
    "images": val_imgs,
    "annotations": [ann for ann in annotations if ann['image_id'] in val_ids],
    "categories": categories
}

test_data = {
    "images": test_imgs,
    "annotations": [ann for ann in annotations if ann['image_id'] in test_ids],
    "categories": categories
}

with open(train_json, 'w') as f: json.dump(train_data, f)
with open(val_json, 'w') as f: json.dump(val_data, f)
with open(test_json, 'w') as f: json.dump(test_data, f)

print(f"Train Set: {len(train_imgs)}, Validation Set: {len(val_imgs)}, Test Set: {len(test_imgs)}")

Train Set: 10500, Validation Set: 2250, Test Set: 2250


In [7]:
!rm dataset/.resume_state.json dataset/annotations.json

In [8]:
import os
import json
import yaml
import shutil
from pathlib import Path
from ultralytics.data.converter import convert_coco

source_path = Path("dataset")
yolo_path = Path("yolo_dataset") 

json_mapping = {
    "train_annotations.json": "train.json",
    "val_annotations.json": "val.json",
    "test_annotations.json": "test.json"
}

for old_name, new_name in json_mapping.items():
    old_path = source_path / old_name
    if old_path.exists():
        old_path.rename(source_path / new_name)

convert_coco(
    labels_dir=str(source_path),
    save_dir=str(yolo_path),
    cls91to80=False,
)

train_json = source_path / "train.json"
if train_json.exists():
    with open(train_json) as f:
        coco = json.load(f)

    names = {i: cat["name"] for i, cat in enumerate(sorted(coco["categories"], key=lambda x: x["id"]))}

    yaml_path = yolo_path / "dataset.yaml"
    with open(yaml_path, "w") as f:
        yaml.dump({
            "path": str(yolo_path.resolve()),
            "train": "images/train",
            "val": "images/val",
            "test": "images/test",
            "names": names
        }, f, default_flow_style=False)
    print(f"Created {yaml_path}")

source_images_dir = source_path / "images"

for split in ["train", "val", "test"]:
    lbl_dir = yolo_path / "labels" / split
    img_split_dir = yolo_path / "images" / split

    if lbl_dir.exists():
        img_split_dir.mkdir(parents=True, exist_ok=True)

        for lbl_file in lbl_dir.glob("*.txt"):
            for ext in ['.jpg', '.jpeg', '.png', '.JPG']:
                img_src = source_images_dir / (lbl_file.stem + ext)
                if img_src.exists():
                    shutil.copy(str(img_src), str(img_split_dir / img_src.name))
                    break

print("Dataset organized in YOLO format.")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Annotations /kaggle/working/dataset/test.json: 100% ━━━━━━━━━━━━ 1691/1691 9.0Kit/s 0.2s
Annotations /kaggle/working/dataset/train.json: 100% ━━━━━━━━━━━━ 7892/7892 9.3Kit/s 0.8s
Annotations /kaggle/working/dataset/val.json: 100% ━━━━━━━━━━━━ 1691/1691 9.7Kit/s 0.2s
COCO data converted successfully.
Results saved to /kaggle/working/yolo_dataset
Created yolo_dataset/dataset.yaml
Dataset organized in YOLO format.


In [9]:
from ultralytics import RTDETR

model = RTDETR("rtdetr-l.pt")

results = model.train(
    data="yolo_dataset/dataset.yaml",
    epochs=50,
    imgsz=640,
    batch=32,
    device=[0,1],
    workers=2,
    freeze=None,
    patience=7,
    name="anime_character_detector"
)

Ultralytics 8.4.48 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
                                                       CUDA:1 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=yolo_dataset/dataset.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=rtdetr-l.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=anime_character_detecto

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution

       1/50      12.3G     0.3663      0.901     0.4354         29        640: 100% ━━━━━━━━━━━━ 247/247 1.2s/it 5:02
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 27/27 1.4it/s 19.2s
                   all       1691       2976      0.827      0.738      0.795      0.469

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size


/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution

       2/50      12.6G     0.3411      0.589     0.3954         41        640: 100% ━━━━━━━━━━━━ 247/247 1.2s/it 4:51
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 27/27 1.7it/s 15.7s
                   all       1691       2976      0.579       0.22      0.211      0.114

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size


...

       6/50      12.6G     0.3578     0.6898     0.4018         40        640: 100% ━━━━━━━━━━━━ 247/247 1.2s/it 4:44
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 27/27 1.8it/s 14.7s
                   all       1691       2976          0          0          0          0
WARNING ⚠️ Skipping checkpoint save at epoch 5: EMA contains NaN/Inf

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size


/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution

       7/50      12.6G     0.3541     0.6824      0.396         41        640: 100% ━━━━━━━━━━━━ 247/247 1.1s/it 4:43
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 27/27 1.8it/s 14.7s
                   all       1691       2976          0          0          0          0
WARNING ⚠️ Skipping checkpoint save at epoch 6: EMA contains NaN/Inf

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size


/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution

       8/50      12.7G     0.3644     0.7432     0.4133         44        640: 100% ━━━━━━━━━━━━ 247/247 1.1s/it 4:44
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 27/27 1.8it/s 14.8s
                   all       1691       2976          0          0          0          0
EarlyStopping: Training stopped early as no improvement observed in last 7 epochs. Best results observed at epoch 1, best model saved as best.pt.
To update EarlyStopping(patience=7) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.
WARNING ⚠️ Skipping checkpoint save at epoch 7: EMA contains NaN/Inf

8 epochs completed in 0.676 hours.
Optimizer stripped from /kaggle/working/runs/detect/anime_character_detector/weights/last.pt, 66.2MB
Optimizer stripped from /kaggle/working/runs/detect/anime_character_detector/weights/best.pt, 66.2MB

Validating /kaggle/working/runs/detect/anime_character_detector/weights/best.pt...

In [10]:
metrics = model.val(split='test')

print(f"mAP50-95: {metrics.box.map}")
print(f"mAP50: {metrics.box.map50}")

Ultralytics 8.4.48 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
rt-detr-l summary: 310 layers, 31,985,795 parameters, 0 gradients, 103.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1955.4±552.4 MB/s, size: 96.5 KB)
val: Scanning /kaggle/working/yolo_dataset/labels/test... 1691 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1691/1691 1.3Kit/s 1.3s
val: New cache created: /kaggle/working/yolo_dataset/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 106/106 1.4it/s 1:14
                   all       1691       2976      0.818      0.751      0.803       0.48
Speed: 0.7ms preprocess, 40.2ms inference, 0.0ms loss, 0.2ms postprocess per image
Results saved to /kaggle/working/runs/detect/val
mAP50-95: 0.47968025190108887
mAP50: 0.8027022028326106


In [11]:
model.export(format="onnx")

Ultralytics 8.4.48 🚀 Python-3.12.12 torch-2.10.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
rt-detr-l summary: 310 layers, 31,985,795 parameters, 0 gradients, 103.4 GFLOPs

PyTorch: starting from '/kaggle/working/runs/detect/anime_character_detector/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 300, 6) (63.1 MB)
requirements: Ultralytics requirements ['onnxslim>=0.1.71', 'onnxruntime-gpu'] not found, attempting AutoUpdate...
Using Python 3.12.12 environment at: /usr
Resolved 12 packages in 334ms
 Downloaded onnxruntime-gpu
Prepared 2 packages in 2.90s
Installed 2 packages in 21ms
 + onnxruntime-gpu==1.26.0
 + onnxslim==0.1.93

requirements: AutoUpdate success ✅ 4.1s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.20.1 opset 20...


/usr/local/lib/python3.12/dist-packages/torch/onnx/_internal/torchscript_exporter/jit_utils.py:305: UserWarning: Constant folding - Only steps=1 can be constant folded for opset >= 10 onnx::Slice op. Constant folding not applied. (Triggered internally at /pytorch/torch/csrc/jit/passes/onnx/constant_fold.cpp:178.)
  _C._jit_pass_onnx_node_shape_type_inference(node, params_dict, opset_version)
/usr/local/lib/python3.12/dist-packages/torch/onnx/_internal/torchscript_exporter/symbolic_opset11.py:954: UserWarning: Exporting aten::index operator of advanced indexing in opset 20 is achieved by combination of multiple ONNX operators, including Reshape, Transpose, Concat, and Gather. If indices include negative values, the exported graph will produce incorrect results.
  return opset9.index(g, self, index)
/usr/local/lib/python3.12/dist-packages/torch/onnx/_internal/torchscript_exporter/utils.py:714: UserWarning: Constant folding - Only steps=1 can be constant folded for opset >= 10 onnx::Slice

ONNX: slimming with onnxslim 0.1.93...
ONNX: export success ✅ 23.6s, saved as '/kaggle/working/runs/detect/anime_character_detector/weights/best.onnx' (125.4 MB)

Export complete (27.1s)
Results saved to /kaggle/working/runs/detect/anime_character_detector/weights/best.onnx
Predict:         yolo predict task=detect model=/kaggle/working/runs/detect/anime_character_detector/weights/best.onnx imgsz=640 
Validate:        yolo val task=detect model=/kaggle/working/runs/detect/anime_character_detector/weights/best.onnx imgsz=640 data=yolo_dataset/dataset.yaml  
Visualize:       https://netron.app


'/kaggle/working/runs/detect/anime_character_detector/weights/best.onnx'

In [12]:
import os
import shutil

KEEP_DIR = "runs"
BASE_DIR = os.getcwd()  

for item in os.listdir(BASE_DIR):
    item_path = os.path.join(BASE_DIR, item)

    if item == KEEP_DIR:
        continue

    try:
        if os.path.isfile(item_path) or os.path.islink(item_path):
            os.remove(item_path)
            print(f"Deleted file: {item_path}")
        elif os.path.isdir(item_path):
            shutil.rmtree(item_path)
            print(f"Deleted folder: {item_path}")
    except Exception as e:
        print(f"Failed to delete {item_path}: {e}")

Deleted file: /kaggle/working/__huggingface_repos__.json
Deleted file: /kaggle/working/__notebook__.ipynb
Deleted folder: /kaggle/working/dataset
Deleted file: /kaggle/working/danbooru-annotated-images.zip
Deleted file: /kaggle/working/yolo26n.pt
Deleted folder: /kaggle/working/yolo_dataset
Deleted file: /kaggle/working/rtdetr-l.pt
